In [18]:
#pip install statsmodels

In [19]:
import os
import re
import warnings
from pathlib import Path
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from patsy import dmatrices
from scipy import stats
from statsmodels.stats.multitest import multipletests
 
warnings.filterwarnings("ignore")

In [ ]:
DATA = Path(os.environ.get("TIKI_DATA", "tiki_du_lieu.xlsx"))
SHEET = "Du_lieu_nghien_cuu" 

OUT = Path("results")
(OUT / "fig").mkdir(parents=True, exist_ok=True)
Y = "sold_log" 
ALPHA = 0.05
MIN_RATED = 5                    
STRONG_DELTA = 0.33              
STRONG_RHO = 0.30                
TIERS = [0, 4, 29, np.inf]
TIER_LABELS = ["0-4 review", "5-29 review", "30+ review"]


# Hàm hỗ trợ

In [ ]:
def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"
 
 
def p_dir(est, p_two, expected):
    """p hai phía CÓ KIỂM TRA HƯỚNG. expected: '+', '-' hoặc None (không có hướng).
    Sai hướng => trả về 1.0 (không ủng hộ), đúng hướng => trả về p hai phía."""
    if expected is None:
        return float(p_two)
    ok = (est > 0) if expected == "+" else (est < 0)
    return float(p_two) if ok else 1.0
 
 
def label_delta(d):
    a = abs(d)
    return "không đáng kể" if a < 0.147 else "nhỏ" if a < 0.33 else "vừa" if a < 0.474 else "lớn"
 
 
def label_rho(r):
    a = abs(r)
    return "rất yếu" if a < 0.1 else "yếu" if a < 0.3 else "vừa" if a < 0.5 else "mạnh"
 
 
def mwu(a, b):
    """Mann-Whitney U HAI PHÍA; a so với b. Cliff's delta = 2*U1/(n1*n2) - 1 (dương nghĩa là a thường lớn hơn b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    ranks = stats.rankdata(np.concatenate([a, b]))
    u1 = ranks[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    delta = 2 * u1 / (len(a) * len(b)) - 1
    p = float(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)
    return dict(n_a=len(a), n_b=len(b), med_a=float(np.median(a)), med_b=float(np.median(b)),
                mean_a=float(a.mean()), mean_b=float(b.mean()), delta=float(delta), p=p, label=label_delta(delta))
 
 
def spearman(x, y):
    """Spearman HAI PHÍA + khoảng tin cậy 95% (Fisher z, hiệu chỉnh Bonett-Wright)."""
    x, y = pd.Series(np.asarray(x, float)), pd.Series(np.asarray(y, float))
    ok = x.notna() & y.notna()
    x, y = x[ok].values, y[ok].values
    res = stats.spearmanr(x, y)
    rho, n = float(res.statistic), len(x)
    se = np.sqrt((1 + rho ** 2 / 2) / (n - 3))
    return dict(n=n, rho=rho, p=float(res.pvalue), lo=float(np.tanh(np.arctanh(rho) - 1.96 * se)),
                hi=float(np.tanh(np.arctanh(rho) + 1.96 * se)), se_z=float(se), label=label_rho(rho))
 
 
def ols_cluster(formula, data):
    """OLS, sai số chuẩn gom theo người bán (seller_id), vì sản phẩm cùng shop không độc lập."""
    _, X = dmatrices(formula, data, return_type="dataframe")
    return smf.ols(formula, data.loc[X.index]).fit(cov_type="cluster", cov_kwds={"groups": data.loc[X.index, "seller_id"]})
 
 
def logit_cluster(formula, data):
    _, X = dmatrices(formula, data, return_type="dataframe")
    return smf.logit(formula, data.loc[X.index]).fit(disp=0, cov_type="cluster",
                                                     cov_kwds={"groups": data.loc[X.index, "seller_id"]})
 
 
def coef_row(res, name, expected=None):
    est, se, p2 = float(res.params[name]), float(res.bse[name]), float(res.pvalues[name])
    ci = res.conf_int().loc[name]
    return dict(est=est, se=se, lo=float(ci[0]), hi=float(ci[1]), p_two=p2, p=p_dir(est, p2, expected),
                n=int(res.nobs), dir_ok=(expected is None) or ((est > 0) == (expected == "+")))
 
 
def lincom(res, weights, expected=None):
    """Kiểm định tổ hợp tuyến tính các hệ số: sum(w_k * beta_k) = 0."""
    v = np.zeros(len(res.params))
    for k, w in weights.items():
        v[list(res.params.index).index(k)] = w
    t = res.t_test(v)
    est, p2, se = float(np.squeeze(t.effect)), float(np.squeeze(t.pvalue)), float(np.squeeze(t.sd))
    return dict(est=est, se=se, lo=est - 1.96 * se, hi=est + 1.96 * se, p_two=p2, p=p_dir(est, p2, expected))
 
 
def wald_interaction(res, prefix):
    names = [n for n in res.params.index if n.startswith(prefix)]
    R = np.zeros((len(names), len(res.params)))
    for i, n in enumerate(names):
        R[i, list(res.params.index).index(n)] = 1
    w = res.wald_test(R, use_f=False, scalar=True)
    return float(w.statistic), float(w.pvalue), len(names)
 
 
def ci_str(lo, hi, d=3):
    return "" if (lo is None or pd.isna(lo)) else f"{lo:.{d}f} … {hi:.{d}f}"
 
 
class Report:
    def __init__(self):
        self.sheets = {}
 
    def add(self, sheet, title, df):
        self.sheets.setdefault(sheet, []).append((title, df))
 
    @staticmethod
    def tidy(df):
        out = df.copy()
        for c in out.columns:
            if str(c) == "p" or str(c).startswith("p_"):
                out[c] = out[c].map(lambda v: fmt_p(v) if isinstance(v, (int, float, np.floating)) else v)
            elif pd.api.types.is_float_dtype(out[c]):
                out[c] = out[c].round(4)
        return out
 
    def save(self, path):
        with pd.ExcelWriter(path, engine="openpyxl") as xw:
            for sheet, blocks in self.sheets.items():
                row = 0
                for title, df in blocks:
                    df = self.tidy(df)
                    df.to_excel(xw, sheet_name=sheet, startrow=row + 1, index=False)
                    ws = xw.sheets[sheet]
                    cell = ws.cell(row=row + 1, column=1)
                    cell.value = title
                    cell.font = cell.font.copy(bold=True, size=12)
                    row += len(df) + 4
                ws = xw.sheets[sheet]
                for col in ws.columns:
                    width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                    ws.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)
                ws.freeze_panes = "A2" if sheet == "Tổng hợp" else None
 
 
REP = Report()
FULLREG = []        
 
 
def keep_reg(name, res):
    ci = res.conf_int()
    t = pd.DataFrame({"hệ_số": res.params, "sai_số_chuẩn": res.bse, "p": res.pvalues,
                      "ci_2.5%": ci[0], "ci_97.5%": ci[1]}).reset_index().rename(columns={"index": "biến"})
    t.loc[len(t)] = ["[n quan sát]", res.nobs, np.nan, np.nan, np.nan, np.nan]
    FULLREG.append((name, t))
 
 
class Tests:
    def __init__(self):
        self.rows = []
 
    def add(self, role, name, n, est, ci="", p=np.nan, note_=""):
        self.rows.append(dict(vai_trò=role, kiểm_định=name, n=n, thống_kê_hoặc_ước_lượng=est, ci95=ci, p_hai_phía=p, nhận_xét=note_))
 
    def df(self):
        return pd.DataFrame(self.rows)

# Đọc dữ liệu, tạo biến, tạo 2 phiên bản dữ liệu

In [ ]:
df = pd.read_excel(DATA, sheet_name=SHEET)
REQUIRED = [
    "product_id", "seller_id", "name", "main_category", "is_tiki_trading", "is_branded",
    "quantity_sold", "sold_log", "log_price_ratio", "rating_average", "review_count",
    "n_images", "has_video", "n_specs", "desc_len",
    "s_is_official", "s_avg_rating_point", "s_total_follower"
]
_missing = [c for c in REQUIRED if c not in df.columns]
if _missing:
    raise KeyError(f"File {DATA.name} thiếu các cột: {_missing}")

# Ép kiểu Boolean cho các cờ phân mẫu 
if "in_core" in df.columns:
    df["in_core"] = df["in_core"].astype(bool)
else:
    df["in_core"] = True

if "in_velocity" in df.columns:
    df["in_velocity"] = df["in_velocity"].astype(bool)

# Tạo các biến phụ trợ dùng cho phân tích 
df["in_rated"] = df.review_count >= MIN_RATED
df["is_branded"] = df.is_branded.fillna(0)

# Phân loại nhóm nhà bán
df["seller_type"] = np.select(
    [df.is_tiki_trading == 1, df.s_is_official == 1],
    ["Tiki Trading", "Official khác"],
    "Seller thường"
)
print(f"[Dữ liệu] {len(df):,} sản phẩm; in_core (mẫu chính) = {int(df.in_core.sum()):,}")

df["sold_filled"] = df.quantity_sold.fillna(0)
df["is_bestseller"] = (
    (df.sold_filled >= df.groupby("main_category").sold_filled.transform(lambda x: x.quantile(0.75)))
    & (df.sold_filled > 0)
).astype(int)

# Rating Bayes & Rating chuẩn hóa theo mẫu chính
BAYES_C = 5
_m_all = df.loc[df.in_core & df.rating_average.notna(), "rating_average"].mean()
df["rating_bayes"] = (
    (BAYES_C * _m_all + df.rating_average * df.review_count) / (BAYES_C + df.review_count)
).where(df.rating_average.notna())

_m22 = df.in_core & df.rating_average.notna()
df["rating_c"] = df.rating_average - df.loc[_m22, "rating_average"].mean()
df["log_rc_c"] = np.log1p(df.review_count) - np.log1p(df.loc[_m22, "review_count"]).mean()
df["enough"] = (df.review_count >= MIN_RATED).astype(int)

core = df[df.in_core].copy()

# Biến kiểm soát OLS
CTRL = "C(main_category) + is_branded"
CTRL_PX = CTRL + " + log_price_ratio"

# 2 phiên bản kiểm chứng độ vững
VARIANT_NAMES = ["Mẫu đầy đủ", "Bỏ Tiki Trading"]

def variant_masks(base=None):
    m = df.in_core if base is None else (df.in_core & base)
    return {
        "Mẫu đầy đủ": m,
        "Bỏ Tiki Trading": m & (df.is_tiki_trading == 0)
    }

SUMMARY, ROBUST = [], []

def robust_ols(hid, formula, keys, expected, base=None):
    """Chạy lại mô hình kiểm chứng trên 2 phiên bản (Mẫu đầy đủ & Bỏ Tiki Trading)."""
    keys = [keys] if isinstance(keys, str) else keys
    for vname, mask in variant_masks(base).items():
        try:
            res = ols_cluster(formula, df[mask])
            rows = [coef_row(res, k, expected) for k in keys]
            ROBUST.append(dict(
                giả_thuyết=hid, 
                phiên_bản=vname, 
                n=rows[0]["n"],
                ước_lượng=" / ".join(f"{r['est']:.3f}" for r in rows),
                p_hai_phía_max=max(r["p_two"] for r in rows),
                đúng_hướng=all(r["dir_ok"] for r in rows),
                đạt=all(r["dir_ok"] and r["p_two"] < ALPHA for r in rows)
            ))
        except Exception as e:
            print(f"[Độ vững {hid}] bỏ qua '{vname}': {e}")

def add_summary(hid, nhom, claim, sample, n, main_test, p_main, effect_txt, strong, conf_test, conf_est, p_conf):
    SUMMARY.append(dict(
        id=hid, nhóm=nhom, giả_thuyết=claim, mẫu=sample, n=n,
        kiểm_định_chính=main_test, p_chính=p_main,
        mức_ảnh_hưởng=effect_txt, hiệu_ứng_không_yếu=strong,
        kiểm_chứng=conf_test, ước_lượng_kiểm_chứng=conf_est, p_kiểm_chứng=p_conf
    ))

[Dữ liệu] 2,034 sản phẩm; in_core (mẫu chính) = 2,034


 # NHÓM 1 — Giá bán sản phẩm


# H1.1  Giá tương đối so với mặt bằng ngành


In [ ]:
print("\n=== H1.1 Giá tương đối so với mặt bằng ngành ===")
T = Tests()
CAT = "main_category"
MIN_CAT_N = 50  


def within_cat_spearman(d, x="log_price_ratio", y=Y):
    """Spearman gộp trong ngành: đổi x và y thành phân vị TRONG TỪNG NGÀNH, gộp các ngành rồi tính tương quan (Pearson trên phân vị)."""
    t = d[[x, y, CAT]].dropna().copy()
    t["rx"] = t.groupby(CAT)[x].rank(pct=True)
    t["ry"] = t.groupby(CAT)[y].rank(pct=True)
    r, p = stats.pearsonr(t.rx, t.ry)
    n = len(t)
    se = np.sqrt((1 + r ** 2 / 2) / (n - 3))
    return dict(
        n=n, rho=float(r), p=float(p),
        lo=float(np.tanh(np.arctanh(r) - 1.96 * se)),
        hi=float(np.tanh(np.arctanh(r) + 1.96 * se)),
        label=label_rho(r)
    )


# Kiểm định Spearman gộp trong ngành
wc = within_cat_spearman(core)
p11a = p_dir(wc["rho"], wc["p"], "-")
T.add(
    "Chính (phần A: hướng chung)",
    "Spearman gộp trong ngành (xếp hạng riêng từng ngành rồi gộp)",
    wc["n"],
    f"rho={wc['rho']:.3f} ({wc['label']})",
    ci_str(wc["lo"], wc["hi"]),
    wc["p"],
    "kỳ vọng âm; p chưa gom theo shop, xem phần kiểm chứng"
)

# Spearman riêng từng ngành 
rows = []
for cat, g in core.groupby(CAT):
    s_ = spearman(g.log_price_ratio, g[Y])
    rows.append(dict(
        ngành=cat, n=s_["n"], spearman_rho=s_["rho"],
        ci95=ci_str(s_["lo"], s_["hi"]), p_thô=s_["p"]
    ))
cat_tbl = pd.DataFrame(rows)
cat_tbl["p_Holm"] = multipletests(cat_tbl.p_thô, method="holm")[1]
cat_tbl["hướng"] = np.where(cat_tbl.spearman_rho < 0, "rẻ -> bán nhiều hơn", "đắt hơn mặt bằng -> bán nhiều hơn")
cat_tbl["ghi_chú"] = np.where(cat_tbl.n < MIN_CAT_N, "ít dữ liệu", "")

k_neg = int(((cat_tbl.spearman_rho < 0) & (cat_tbl.p_Holm < ALPHA)).sum())
k_pos = int(((cat_tbl.spearman_rho > 0) & (cat_tbl.p_Holm < ALPHA)).sum())
T.add(
    "Tham khảo",
    f"Số ngành có rho ÂM có ý nghĩa sau Holm (trong {len(cat_tbl)} ngành)",
    int(cat_tbl.n.sum()),
    f"{k_neg}",
    "",
    np.nan,
    f"{k_pos} ngành rho DƯƠNG có ý nghĩa; {len(cat_tbl) - k_neg - k_pos} ngành không có ý nghĩa"
)

# OLS Tương tác kiểm tra độ dốc giữa các ngành
ols_y =  Y
f11i = f"{ols_y} ~ log_price_ratio * C(main_category) + is_branded"
res11i = ols_cluster(f11i, core)
chi2_w, p_w, k_w = wald_interaction(res11i, "log_price_ratio:")
keep_reg("H1.1 OLS tương tác giá x ngành", res11i)
T.add(
    "Chính (phần B: khác nhau giữa ngành)",
    f"Wald chung {k_w} hệ số tương tác giá x ngành",
    int(res11i.nobs),
    f"chi2={chi2_w:.1f}",
    "",
    p_w,
    "p nhỏ = độ dốc của giá khác nhau giữa các ngành"
)

# OLS Kiểm chứng đơn giản
f11 = f"{ols_y} ~ log_price_ratio + {CTRL}"
res11 = ols_cluster(f11, core)
c11 = coef_row(res11, "log_price_ratio", "-")
keep_reg("H1.1 OLS sold_log ~ log_price_ratio", res11)
T.add(
    "Kiểm chứng",
    "OLS gom theo shop, kiểm soát ngành (hệ số log_price_ratio, kỳ vọng âm)",
    c11["n"],
    f"{c11['est']:.3f} (giá gấp đôi mặt bằng ngành ứng với doanh số x{np.exp(c11['est'] * np.log(2)):.2f})",
    ci_str(c11["lo"], c11["hi"]),
    c11["p_two"],
    ""
)

# Phân tích độ nhạy theo Tiki Trading
nt_ = core[core.is_tiki_trading == 0]
tt_ = core[core.is_tiki_trading == 1]
wc_nt, wc_tt = within_cat_spearman(nt_), within_cat_spearman(tt_)

T.add(
    "Độ nhạy", "Bỏ Tiki Trading: Spearman gộp trong ngành", wc_nt["n"],
    f"rho={wc_nt['rho']:.3f} ({wc_nt['label']})", ci_str(wc_nt["lo"], wc_nt["hi"]), wc_nt["p"], ""
)
T.add(
    "Độ nhạy", "Chỉ Tiki Trading: Spearman gộp trong ngành", wc_tt["n"],
    f"rho={wc_tt['rho']:.3f}", ci_str(wc_tt["lo"], wc_tt["hi"]), wc_tt["p"],
    f"Tiki Trading chiếm {len(tt_)}/{len(core)} sản phẩm"
)

res11i_nt = ols_cluster(f11i, nt_)
_, p_w_nt, _ = wald_interaction(res11i_nt, "log_price_ratio:")
T.add("Độ nhạy", "Bỏ Tiki Trading: Wald tương tác giá x ngành", int(res11i_nt.nobs), "", "", p_w_nt, "")


REP.add("Nhóm 1", "H1.1 — Sản phẩm rẻ hơn mặt bằng ngành bán nhiều hơn; ảnh hưởng của giá khác nhau giữa các ngành", T.df())
REP.add("Nhóm 1", "H1.1 — Spearman trong từng ngành (Holm cho các ngành)", cat_tbl)
robust_ols("H1.1", f11, "log_price_ratio", "-")

print(
    f"[H1.1] rho gộp trong ngành={wc['rho']:.3f} (p={fmt_p(wc['p'])}); "
    f"Wald p={fmt_p(p_w)}; OLS p={fmt_p(c11['p_two'])}; "
    f"{k_neg} ngành âm / {k_pos} ngành dương có ý nghĩa sau Holm"
)

add_summary(
    "H1.1", 1, "Rẻ hơn mặt bằng ngành bán nhiều hơn; ảnh hưởng của giá khác nhau giữa các ngành", "in_core", wc["n"],
    "Spearman gộp trong ngành + Wald tương tác (p = max)", max(p11a, p_w),
    f"rho gộp={wc['rho']:.2f} ({wc['label']}); ngành: {k_neg} âm, {k_pos} dương", abs(wc["rho"]) >= STRONG_RHO,
    "OLS log_price_ratio (kiểm soát ngành)", f"{c11['est']:.3f}", c11["p"]
)


=== H1.1 Giá tương đối so với mặt bằng ngành ===
[H1.1] rho gộp trong ngành=-0.052 (p=0.018); Wald p=<0.001; OLS p=0.424; 1 ngành âm / 1 ngành dương có ý nghĩa sau Holm



# NHÓM 2 — Đánh giá khách hàng

#   H2.1 Rating cao bán nhiều hơn 
#   H2.2 Rating chỉ giúp khi đủ review

In [ ]:
print("\n=== H2.1 Rating (hiệu chỉnh Bayes) cao -> bán nhiều ===")
T = Tests()
ols_y =  Y

rated = df[df.in_core & df.in_rated].copy()
sp21 = spearman(rated.rating_bayes, rated[Y])
p21 = p_dir(sp21["rho"], sp21["p"], "+")

T.add(
    "Chính", "Spearman(rating_bayes, doanh số)", sp21["n"],
    f"rho={sp21['rho']:.3f} ({sp21['label']})", ci_str(sp21["lo"], sp21["hi"]),
    sp21["p"], f"mẫu in_core có từ {MIN_RATED} review; {(rated.rating_bayes >= 4.8).mean():.0%} sản phẩm có rating_bayes >= 4,8"
)

f21 = f"{ols_y} ~ rating_bayes + {CTRL_PX}"
res21 = ols_cluster(f21, rated)
c21 = coef_row(res21, "rating_bayes", "+")
keep_reg("H2.1 OLS sold_log ~ rating_bayes (không kiểm soát review)", res21)

T.add(
    "Kiểm chứng", "OLS không kiểm soát review (mô hình không có nhóm hệ quả)", c21["n"],
    f"{c21['est']:.3f} (+0,1 sao ≈ +{(np.exp(c21['est'] * 0.1) - 1):.1%})", ci_str(c21["lo"], c21["hi"]), c21["p_two"], ""
)

f21a = f"{ols_y} ~ rating_bayes + np.log1p(review_count) + {CTRL_PX}"
res21a = ols_cluster(f21a, rated)
c21a = coef_row(res21a, "rating_bayes", "+")
keep_reg("H2.1 OLS sold_log ~ rating_bayes + np.log1p(review_count) (có nhóm hệ quả)", res21a)

T.add(
    "Độ nhạy", "OLS CÓ kiểm soát log review (mô hình có nhóm hệ quả)", c21a["n"],
    f"{c21a['est']:.3f}", ci_str(c21a["lo"], c21a["hi"]),
    c21a["p_two"], "một phần tác động của rating đi qua số review"
)

hi_g, lo_g = rated[rated.rating_bayes >= 4.8], rated[rated.rating_bayes < 4.8]
rm = mwu(hi_g[Y], lo_g[Y])

T.add(
    "Độ nhạy", "So sánh nhóm rating_bayes >= 4,8 với dưới 4,8: Mann-Whitney", rm["n_a"] + rm["n_b"],
    f"delta={rm['delta']:.3f} ({rm['label']})", "", rm["p"], f"n = {rm['n_a']} so với {rm['n_b']}"
)

REP.add("Nhóm 2", "H2.1 — Sản phẩm điểm cao hơn thì bán nhiều hơn", T.df())
robust_ols("H2.1", f21, "rating_bayes", "+", base=df.in_rated)

add_summary(
    "H2.1", 2, "Rating (hiệu chỉnh Bayes) cao hơn thì bán nhiều hơn (liên quan)", f"in_core & review>={MIN_RATED}", sp21["n"], "Spearman",
    p21, f"rho={sp21['rho']:.2f} ({sp21['label']})", abs(sp21["rho"]) >= STRONG_RHO,
    "OLS rating_bayes (không kiểm soát review)", f"{c21['est']:.3f}", c21["p"]
)


print("\n=== H2.2 Rating chỉ giúp khi đủ review ===")
T = Tests()
ols_y =  Y

_m22 = df.in_core & df.rating_average.notna()
d22 = df[_m22].copy()

# Tính lại biến chuẩn hóa điểm và log số review từ cột gốc
mean_lrc = np.log1p(d22.review_count).mean()
d22["log_rc_c"] = np.log1p(d22.review_count) - mean_lrc
d22["rating_c"] = d22.rating_average - d22.rating_average.mean()
d22["tier"] = pd.cut(d22.review_count, TIERS, labels=TIER_LABELS, include_lowest=True)

#  OLS Tương tác liên tục
f22 = f"{ols_y} ~ rating_c * log_rc_c + {CTRL_PX}"
res22 = ols_cluster(f22, d22)
KI = "rating_c:log_rc_c"
c22 = coef_row(res22, KI, "+")
keep_reg("H2.2 OLS tương tác liên tục rating x log(review)", res22)

T.add(
    "Chính", "OLS tương tác rating x log(1 + review)", c22["n"], f"{c22['est']:.3f}",
    ci_str(c22["lo"], c22["hi"]), c22["p_two"],
    "hệ số dương = rating có tác dụng lớn hơn khi sản phẩm có nhiều review hơn"
)

# Tác dụng biên của rating tại các mức số review
marg = []
for rc in [1, 3, 5, 10, 30, 100, 300]:
    m_ = lincom(res22, {"rating_c": 1, KI: np.log1p(rc) - mean_lrc})
    marg.append(dict(
        số_review=rc, độ_dốc_rating=m_["est"],
        ci_thấp=m_["lo"], ci_cao=m_["hi"], p_hai_phía=m_["p_two"]
    ))
marg = pd.DataFrame(marg)

# Logit Kiểm chứng bán chạy top 25% ngành
f22l = f"is_bestseller ~ rating_c * log_rc_c + {CTRL_PX}"
lg22 = logit_cluster(f22l, d22)
lg_est, lg_p2 = float(lg22.params[KI]), float(lg22.pvalues[KI])

T.add(
    "Kiểm chứng", "Logit 'bán chạy' (top 25% ngành): tương tác liên tục", int(lg22.nobs),
    f"hệ số={lg_est:.3f} (OR={np.exp(lg_est):.2f})",
    ci_str(*lg22.conf_int().loc[KI]), lg_p2, ""
)

#  OLS Tương tác theo ngưỡng đủ review
f22t = f"{ols_y} ~ rating_c * enough + log_rc_c + {CTRL_PX}"
res22t = ols_cluster(f22t, d22)
KT = "rating_c:enough"
c22t = coef_row(res22t, KT, "+")
keep_reg("H2.2 OLS tương tác theo ngưỡng 'đủ review' (độ nhạy)", res22t)

T.add(
    "Độ nhạy", f"OLS tương tác rating x 'đủ review' (>= {MIN_RATED}), thay cho dạng liên tục",
    c22t["n"], f"{c22t['est']:.3f}", ci_str(c22t["lo"], c22t["hi"]), c22t["p_two"],
    "dạng ngưỡng; báo cáo cả hai dạng"
)

#  So sánh Spearman theo từng nhóm review 
tier_rows = []
for tn in TIER_LABELS:
    g = d22[d22.tier == tn]
    s_ = spearman(g.rating_average, g[Y])
    tier_rows.append(dict(
        nhóm_review=tn, n=s_["n"], spearman_rho=s_["rho"],
        ci95=ci_str(s_["lo"], s_["hi"]), p_hai_phía=s_["p"], se_z=s_["se_z"]
    ))
tier_tbl = pd.DataFrame(tier_rows)

z = (np.arctanh(tier_tbl.spearman_rho.iloc[-1]) - np.arctanh(tier_tbl.spearman_rho.iloc[0])) / np.sqrt(tier_tbl.se_z.iloc[-1] ** 2 + tier_tbl.se_z.iloc[0] ** 2)
p_fz = float(2 * stats.norm.sf(abs(z)))

T.add(
    "Độ nhạy", "So sánh rho nhóm '30+' với '0-4 review' (Fisher z)",
    int(tier_tbl.n.iloc[0] + tier_tbl.n.iloc[-1]), f"z={z:.2f}", "", p_fz, ""
)

# Tổng hợp 
REP.add("Nhóm 2", "H2.2 — Rating cao chỉ giúp bán chạy khi có đủ review", T.df())
REP.add("Nhóm 2", "H2.2 — Tác dụng của +1 sao rating tại từng mức số review (mô hình liên tục)", marg)
REP.add("Nhóm 2", "H2.2 — Spearman(rating, doanh số) theo nhóm số review", tier_tbl.drop(columns="se_z"))

robust_ols("H2.2", f22, KI, "+", base=df.rating_average.notna())

print(
    f"[H2.2] Tương tác liên tục: {c22['est']:.3f} (p hai phía={fmt_p(c22['p_two'])}); "
    f"logit p={fmt_p(lg_p2)}; tương tác ngưỡng p={fmt_p(c22t['p_two'])}"
)

add_summary(
    "H2.2", 2, "Rating cao chỉ giúp bán chạy khi có đủ review (liên quan)", "in_core & review>=1", c22["n"],
    "OLS tương tác rating x log(review)", c22["p"],
    f"dốc +1 sao: {marg.iloc[1].độ_dốc_rating:.2f} (3 review), {marg.iloc[5].độ_dốc_rating:.2f} (100 review)", None,
    "Logit 'bán chạy' tương tác", f"{lg_est:.3f}", p_dir(lg_est, lg_p2, "+")
)


=== H2.1 Rating (hiệu chỉnh Bayes) cao -> bán nhiều ===

=== H2.2 Rating chỉ giúp khi đủ review ===
[H2.2] Tương tác liên tục: 0.029 (p hai phía=0.494); logit p=0.209; tương tác ngưỡng p=0.035


# NHÓM 3 — Chất lượng thông tin sản phẩm
# - H3.1 nhiều ảnh: Spearman(log số ảnh, log doanh số) + OLS.
# - H3.2 có video: Mann-Whitney + OLS.
# - H3.3 thông số kỹ thuật VÀ mô tả chi tiết: cả hai thành phần đều phải ủng hộ (p kết hợp = max); OLS đưa cả hai vào cùng mô hình.

In [ ]:
ols_y =  Y

# H3.1 Nhiều ảnh 

print("\n=== H3.1 Nhiều ảnh ===")
T = Tests()
sp31 = spearman(core.n_images, core[Y])
p31 = p_dir(sp31["rho"], sp31["p"], "+")

T.add(
    "Chính", "Spearman(số ảnh, doanh số)", sp31["n"],
    f"rho={sp31['rho']:.3f} ({sp31['label']})", ci_str(sp31["lo"], sp31["hi"]), sp31["p"],
    f"số ảnh: trung vị {core.n_images.median():.0f}, tối đa {core.n_images.max():.0f}"
)

f31 = f"{ols_y} ~ n_images + {CTRL_PX}"
res31 = ols_cluster(f31, core)
c31 = coef_row(res31, "n_images", "+")
keep_reg("H3.1 OLS sold_log ~ n_images", res31)

T.add(
    "Kiểm chứng", "OLS có kiểm soát (hệ số số ảnh)", c31["n"],
    f"{c31['est']:.3f}", ci_str(c31["lo"], c31["hi"]), c31["p_two"], ""
)

REP.add("Nhóm 3", "H3.1 — Nhiều ảnh bán nhiều hơn", T.df())
robust_ols("H3.1", f31, "n_images", "+")

add_summary(
    "H3.1", 3, "Nhiều ảnh thì bán nhiều hơn", "in_core", sp31["n"],
    "Spearman", p31, f"rho={sp31['rho']:.2f} ({sp31['label']})",
    abs(sp31["rho"]) >= STRONG_RHO, "OLS n_images", f"{c31['est']:.3f}", c31["p"]
)


#  H3.2 Có video 

print("\n=== H3.2 Có video ===")
T = Tests()
v1, v0 = core[core.has_video == 1], core[core.has_video == 0]
r32 = mwu(v1[Y], v0[Y])
p32 = p_dir(r32["delta"], r32["p"], "+")

T.add(
    "Chính", "Mann-Whitney (có video vs không)", r32["n_a"] + r32["n_b"],
    f"delta={r32['delta']:.3f} ({r32['label']})", "", r32["p"],
    f"{core.has_video.mean():.0%} sản phẩm có video; trung vị doanh số {v1.quantity_sold.median():.0f} vs {v0.quantity_sold.median():.0f}"
)

f32 = f"{ols_y} ~ has_video + {CTRL_PX}"
res32 = ols_cluster(f32, core)
c32 = coef_row(res32, "has_video", "+")
keep_reg("H3.2 OLS sold_log ~ has_video", res32)

T.add(
    "Kiểm chứng", "OLS có kiểm soát (hệ số has_video)", c32["n"],
    f"{c32['est']:.3f} (gấp {np.exp(c32['est']):.2f} lần)", ci_str(c32["lo"], c32["hi"]),
    c32["p_two"], ""
)

REP.add("Nhóm 3", "H3.2 — Có video bán nhiều hơn", T.df())
robust_ols("H3.2", f32, "has_video", "+")

add_summary(
    "H3.2", 3, "Có video thì bán nhiều hơn", "in_core", r32["n_a"] + r32["n_b"],
    "Mann-Whitney", p32, f"delta={r32['delta']:.2f} ({r32['label']})",
    abs(r32["delta"]) >= STRONG_DELTA, "OLS has_video", f"{c32['est']:.3f}", c32["p"]
)


#  H3.3 Thông số kỹ thuật & Mô tả 
print("\n=== H3.3 Thông số kỹ thuật và mô tả chi tiết ===")
T = Tests()
spA, spB = spearman(core.n_specs, core[Y]), spearman(core.desc_len, core[Y])
pA, pB = p_dir(spA["rho"], spA["p"], "+"), p_dir(spB["rho"], spB["p"], "+")

T.add(
    "Chính", "Spearman(số thông số, doanh số)", spA["n"],
    f"rho={spA['rho']:.3f} ({spA['label']})", ci_str(spA["lo"], spA["hi"]), spA["p"], ""
)
T.add(
    "Chính", "Spearman(độ dài mô tả, doanh số)", spB["n"],
    f"rho={spB['rho']:.3f} ({spB['label']})", ci_str(spB["lo"], spB["hi"]), spB["p"],
    "giả thuyết chỉ được ủng hộ khi CẢ HAI thành phần ủng hộ"
)

f33 = f"{ols_y} ~ n_specs + desc_len + {CTRL_PX}"
res33 = ols_cluster(f33, core)
cA, cB = coef_row(res33, "n_specs", "+"), coef_row(res33, "desc_len", "+")
keep_reg("H3.3 OLS sold_log ~ n_specs + desc_len", res33)

T.add("Kiểm chứng", "OLS (cùng mô hình): hệ số số thông số", cA["n"], f"{cA['est']:.3f}", ci_str(cA["lo"], cA["hi"]), cA["p_two"], "")
T.add("Kiểm chứng", "OLS (cùng mô hình): hệ số độ dài mô tả", cB["n"], f"{cB['est']:.3f}", ci_str(cB["lo"], cB["hi"]), cB["p_two"], "")

REP.add("Nhóm 3", "H3.3 — Thông tin kỹ thuật và mô tả chi tiết bán nhiều hơn", T.df())
robust_ols("H3.3", f33, ["n_specs", "desc_len"], "+")

add_summary(
    "H3.3", 3, "Thông số kỹ thuật và mô tả chi tiết thì bán nhiều hơn", "in_core", spA["n"],
    "Spearman x 2 (p = max)", max(pA, pB),
    f"rho thông số={spA['rho']:.2f}, rho mô tả={spB['rho']:.2f}",
    min(abs(spA["rho"]), abs(spB["rho"])) >= STRONG_RHO,
    "OLS cả hai biến (p = max)", f"{cA['est']:.3f} / {cB['est']:.3f}", max(cA["p"], cB["p"])
)


=== H3.1 Nhiều ảnh ===

=== H3.2 Có video ===

=== H3.3 Thông số kỹ thuật và mô tả chi tiết ===


#  NHÓM 4 — Đặc điểm nhà bán
# - H4.1 chính hãng: Mann-Whitney + OLS.  
# - H4.2 uy tín (điểm cao)
# - H4.3 nhiều người theo dõi

In [ ]:
ols_y =  Y


def shop_level(var):
    """Hàm tính Spearman ở cấp độ shop (mỗi shop 1 dòng)."""
    g = core.groupby("seller_id").agg(y=(Y, "mean"), x=(var, "first"))
    return spearman(g.x, g.y)



#  H4.1 Cửa hàng chính hãng

print("\n=== H4.1 Cửa hàng chính hãng ===")
T = Tests()
o1, o0 = core[core.s_is_official == 1], core[core.s_is_official == 0]
r41 = mwu(o1[Y], o0[Y])
p41 = p_dir(r41["delta"], r41["p"], "+")

T.add(
    "Chính", "Mann-Whitney (chính hãng vs không)", r41["n_a"] + r41["n_b"],
    f"delta={r41['delta']:.3f} ({r41['label']})", "", r41["p"],
    f"trung vị doanh số {o1.quantity_sold.median():.0f} vs {o0.quantity_sold.median():.0f}; chính hãng gồm cả Tiki Trading"
)

f41 = f"{ols_y} ~ s_is_official + {CTRL_PX}"
res41 = ols_cluster(f41, core)
c41 = coef_row(res41, "s_is_official", "+")
keep_reg("H4.1 OLS sold_log ~ s_is_official", res41)

T.add(
    "Kiểm chứng", "OLS có kiểm soát (hệ số s_is_official)", c41["n"],
    f"{c41['est']:.3f} (gấp {np.exp(c41['est']):.2f} lần)", ci_str(c41["lo"], c41["hi"]),
    c41["p_two"], ""
)

sl = shop_level("s_is_official")
T.add("Độ nhạy", "Cấp shop (mỗi shop một dòng): Spearman", sl["n"], f"rho={sl['rho']:.3f}", ci_str(sl["lo"], sl["hi"]), sl["p"], "")

REP.add("Nhóm 4", "H4.1 — Cửa hàng chính hãng bán nhiều hơn", T.df())
robust_ols("H4.1", f41, "s_is_official", "+")

add_summary(
    "H4.1", 4, "Cửa hàng chính hãng bán nhiều hơn", "in_core", r41["n_a"] + r41["n_b"],
    "Mann-Whitney", p41, f"delta={r41['delta']:.2f} ({r41['label']})",
    abs(r41["delta"]) >= STRONG_DELTA, "OLS s_is_official", f"{c41['est']:.3f}", c41["p"]
)


# H4.2 Người bán uy tín 
print("\n=== H4.2 Người bán uy tín (điểm cao, nhiều theo dõi/đánh giá) ===")
T = Tests()
# Xác định cột quy mô shop 
s_rev_col = "shop_review_log" if "shop_review_log" in core.columns else "s_total_follower"

spR, spN = spearman(core.s_avg_rating_point, core[Y]), spearman(core[s_rev_col], core[Y])
pR, pN = p_dir(spR["rho"], spR["p"], "+"), p_dir(spN["rho"], spN["p"], "+")

T.add("Chính", "Spearman(điểm đánh giá của shop, doanh số)", spR["n"], f"rho={spR['rho']:.3f} ({spR['label']})", ci_str(spR["lo"], spR["hi"]), spR["p"], "")
T.add(
    "Chính", f"Spearman({s_rev_col} của shop, doanh số)", spN["n"], f"rho={spN['rho']:.3f} ({spN['label']})",
    ci_str(spN["lo"], spN["hi"]), spN["p"], "giả thuyết chỉ được ủng hộ khi CẢ HAI thành phần ủng hộ"
)

f42 = f"{ols_y} ~ s_avg_rating_point + {s_rev_col} + {CTRL_PX}"
res42 = ols_cluster(f42, core)
cR, cN = coef_row(res42, "s_avg_rating_point", "+"), coef_row(res42, s_rev_col, "+")
keep_reg(f"H4.2 OLS sold_log ~ s_avg_rating_point + {s_rev_col}", res42)

T.add("Kiểm chứng", "OLS (cùng mô hình): hệ số điểm đánh giá shop", cR["n"], f"{cR['est']:.3f}", ci_str(cR["lo"], cR["hi"]), cR["p_two"], "")
T.add("Kiểm chứng", f"OLS (cùng mô hình): hệ số {s_rev_col} shop", cN["n"], f"{cN['est']:.3f}", ci_str(cN["lo"], cN["hi"]), cN["p_two"], "")

for var, lab in [("s_avg_rating_point", "điểm"), (s_rev_col, "quy mô shop")]:
    s_ = shop_level(var)
    T.add("Độ nhạy", f"Cấp shop: Spearman ({lab})", s_["n"], f"rho={s_['rho']:.3f}", ci_str(s_["lo"], s_["hi"]), s_["p"], "")

REP.add("Nhóm 4", "H4.2 — Người bán uy tín hơn bán nhiều hơn", T.df())
robust_ols("H4.2", f42, ["s_avg_rating_point", s_rev_col], "+")

add_summary(
    "H4.2", 4, "Người bán uy tín hơn (điểm cao, quy mô lớn) bán nhiều hơn", "in_core", spR["n"],
    "Spearman x 2 (p = max)", max(pR, pN),
    f"rho điểm={spR['rho']:.2f}, rho quy mô={spN['rho']:.2f}",
    min(abs(spR["rho"]), abs(spN["rho"])) >= STRONG_RHO,
    "OLS cả hai biến (p = max)", f"{cR['est']:.3f} / {cN['est']:.3f}", max(cR["p"], cN["p"])
)


#  H4.3 Nhiều người theo dõi 

print("\n=== H4.3 Nhiều người theo dõi ===")
T = Tests()
sp43 = spearman(core.s_total_follower, core[Y])
p43 = p_dir(sp43["rho"], sp43["p"], "+")

T.add("Chính", "Spearman(người theo dõi shop, doanh số)", sp43["n"], f"rho={sp43['rho']:.3f} ({sp43['label']})", ci_str(sp43["lo"], sp43["hi"]), sp43["p"], "")

f43 = f"{ols_y} ~ s_total_follower + {CTRL_PX}"
res43 = ols_cluster(f43, core)
c43 = coef_row(res43, "s_total_follower", "+")
keep_reg("H4.3 OLS sold_log ~ s_total_follower", res43)

T.add(
    "Kiểm chứng", "OLS có kiểm soát (hệ số s_total_follower)", c43["n"],
    f"{c43['est']:.3f}", ci_str(c43["lo"], c43["hi"]), c43["p_two"], ""
)

sl = shop_level("s_total_follower")
T.add("Độ nhạy", "Cấp shop (mỗi shop một dòng): Spearman", sl["n"], f"rho={sl['rho']:.3f}", ci_str(sl["lo"], sl["hi"]), sl["p"], "")

REP.add("Nhóm 4", "H4.3 — Người bán có nhiều người theo dõi bán nhiều hơn", T.df())
robust_ols("H4.3", f43, "s_total_follower", "+")

add_summary(
    "H4.3", 4, "Người bán có nhiều người theo dõi bán nhiều hơn", "in_core", sp43["n"],
    "Spearman", p43, f"rho={sp43['rho']:.2f} ({sp43['label']})",
    abs(sp43["rho"]) >= STRONG_RHO, "OLS s_total_follower", f"{c43['est']:.3f}", c43["p"]
)


=== H4.1 Cửa hàng chính hãng ===

=== H4.2 Người bán uy tín (điểm cao, nhiều theo dõi/đánh giá) ===

=== H4.3 Nhiều người theo dõi ===


# Hồi quy chạy hại lần: không có và có nhóm hệ quả

In [ ]:
print("\n=== Mô hình hồi quy tổng hợp (hai lần: không có / có nhóm hệ quả) ===")


JOINT_STABLE = [
    "log_price_ratio", "n_images", "has_video", 
    "desc_len", "n_specs", "s_is_official", "s_avg_rating_point"
]
JOINT_CONSEQ = ["review_count", "rating_average", "s_total_follower"]

ols_y =  Y

# Lọc mẫu sạch dữ liệu
common = core.dropna(subset=JOINT_STABLE + JOINT_CONSEQ + [ols_y, "seller_id"]).copy()
jrows = []
samples = [("Bỏ Tiki Trading", common[common.is_tiki_trading == 0], "")]
samples.append(("Giữ Tiki Trading, kèm cờ is_tiki_trading", common, " + is_tiki_trading"))

for sname, sdata, extra in samples:
    z_ = sdata.copy()
    
    # Chuẩn hóa Z-score cho từng biến độc lập
    for c_ in JOINT_STABLE + JOINT_CONSEQ:
        s_val = z_[c_].astype(float)
        z_[c_] = (s_val - s_val.mean()) / (s_val.std() or 1)
        
    for mname, xs in [
        ("Mô hình B (không có nhóm hệ quả)", JOINT_STABLE), 
        ("Mô hình A (có nhóm hệ quả)", JOINT_STABLE + JOINT_CONSEQ)
    ]:
        res_ = ols_cluster(f"{ols_y} ~ " + " + ".join(xs) + " + " + CTRL + extra, z_)
        ci_ = res_.conf_int()
        
        for v_ in xs:
            jrows.append(dict(
                mẫu=sname, 
                mô_hình=mname, 
                biến=v_, 
                hệ_số_chuẩn_hóa=float(res_.params[v_]),
                sai_số_chuẩn=float(res_.bse[v_]), 
                ci_thấp=float(ci_.loc[v_, 0]), 
                ci_cao=float(ci_.loc[v_, 1]),
                p=float(res_.pvalues[v_]), 
                n=int(res_.nobs), 
                R2=float(res_.rsquared)
            ))

JOINT = pd.DataFrame(jrows)
REP.add("Mô hình tổng hợp", "Mô hình tổng hợp: hệ số chuẩn hóa khi không có và có nhóm hệ quả (sai số gom theo shop)", JOINT)

for (sname, mname), g_ in JOINT.groupby(["mẫu", "mô_hình"], sort=False):
    print(f"  {sname} | {mname}: n={int(g_.n.iloc[0])}, R²={g_.R2.iloc[0]:.2f}")


=== Mô hình hồi quy tổng hợp (hai lần: không có / có nhóm hệ quả) ===
  Bỏ Tiki Trading | Mô hình B (không có nhóm hệ quả): n=1345, R²=0.15
  Bỏ Tiki Trading | Mô hình A (có nhóm hệ quả): n=1345, R²=0.56
  Giữ Tiki Trading, kèm cờ is_tiki_trading | Mô hình B (không có nhóm hệ quả): n=2034, R²=0.38
  Giữ Tiki Trading, kèm cờ is_tiki_trading | Mô hình A (có nhóm hệ quả): n=2034, R²=0.55


# Tổng hợp: Holm, độ vững qua 2 phiên bản

In [30]:
S = pd.DataFrame(SUMMARY)
S["p_chính_Holm"] = multipletests(S.p_chính, method="holm")[1]
S["p_kiểm_chứng_Holm"] = multipletests(S.p_kiểm_chứng, method="holm")[1]
RB = pd.DataFrame(ROBUST)


def robust_label(hid, base):
    """Nhãn độ vững tự động dựa trên so sánh giữa Mẫu đầy đủ và Bỏ Tiki Trading."""
    g = RB[RB.giả_thuyết == hid].set_index("phiên_bản")   
    main = bool(g.loc["Mẫu đầy đủ", "đạt"])
    diff = [v for v in VARIANT_NAMES[1:] if v in g.index and bool(g.loc[v, "đạt"]) != main]
    
    if diff:
        return "Nhạy: kết quả đổi khi " + ", ".join(diff)
    if main:
        return ("Vững: phép kiểm chứng đạt ở cả 2 phiên bản" if base.startswith("Ủng hộ")
                else "Ổn định: p thô có ý nghĩa ở cả 2 phiên bản nhưng KHÔNG qua hiệu chỉnh Holm")
    return "Ổn định: phép kiểm chứng không đạt ở cả 2 phiên bản"


def verdict(r):
    """Phán quyết 3 lớp tự động theo kết quả kiểm định."""
    a, b = r.p_chính_Holm < ALPHA, r.p_kiểm_chứng_Holm < ALPHA
    if a and b:
        base = "Ủng hộ" if (r.hiệu_ứng_không_yếu is not False) else "Ủng hộ yếu"
    elif a or b:
        base = "Ủng hộ một phần"
    else:
        base = "Không ủng hộ"
    return base


S["kết_luận"] = S.apply(verdict, axis=1)
S["độ_vững"] = [robust_label(h, k) for h, k in zip(S.id, S.kết_luận)]
S["kết_luận_cuối"] = S.kết_luận + " — " + S.độ_vững
S["hiệu_ứng_đủ_mạnh"] = S.hiệu_ứng_không_yếu.map({True: "có", False: "yếu (nhỏ)", None: "n/a"})

summary_cols = [
    "id", "giả_thuyết", "mẫu", "n", "kiểm_định_chính", "p_chính", "p_chính_Holm", 
    "mức_ảnh_hưởng", "hiệu_ứng_đủ_mạnh", "kiểm_chứng", "ước_lượng_kiểm_chứng", 
    "p_kiểm_chứng", "p_kiểm_chứng_Holm", "kết_luận", "độ_vững", "kết_luận_cuối"
]

REP.sheets = {
    "Tổng hợp": [(f"Tổng hợp giả thuyết — thước đo: doanh số tích lũy (quantity_sold)", S[summary_cols])],
    **REP.sheets
}

RB_show = RB.copy()
RB_show = RB_show.rename(columns={"p_hai_phía_max": "p_hai_phía"})
REP.add("Độ vững", "Kết quả mô hình kiểm chứng trên 2 phiên bản dữ liệu ('đạt' = đúng hướng và p hai phía < 0,05)", RB_show)

for name, t in FULLREG:
    REP.add("Hồi quy đầy đủ", name, t)


print("\n" + "=" * 70 + "\nTỔNG HỢP GIẢ THUYẾT\n" + "=" * 70)
show = S[["id", "p_chính_Holm", "p_kiểm_chứng_Holm", "mức_ảnh_hưởng", "kết_luận_cuối"]].copy()
show["p_chính_Holm"] = show.p_chính_Holm.map(fmt_p)
show["p_kiểm_chứng_Holm"] = show.p_kiểm_chứng_Holm.map(fmt_p)
print(show.to_string(index=False))


TỔNG HỢP GIẢ THUYẾT
  id p_chính_Holm p_kiểm_chứng_Holm                                  mức_ảnh_hưởng                                                         kết_luận_cuối
H1.1        0.055             0.848  rho gộp=-0.05 (rất yếu); ngành: 1 âm, 1 dương                  Không ủng hộ — Nhạy: kết quả đổi khi Bỏ Tiki Trading
H2.1       <0.001            <0.001                                rho=0.62 (mạnh)                   Ủng hộ — Vững: phép kiểm chứng đạt ở cả 2 phiên bản
H2.2        0.988             0.626 dốc +1 sao: 0.05 (3 review), 0.14 (100 review)                  Không ủng hộ — Nhạy: kết quả đổi khi Bỏ Tiki Trading
H3.1        0.023             0.019                             rho=0.06 (rất yếu)                    Ủng hộ yếu — Nhạy: kết quả đổi khi Bỏ Tiki Trading
H3.2        1.000             0.848                    delta=-0.09 (không đáng kể)                  Không ủng hộ — Nhạy: kết quả đổi khi Bỏ Tiki Trading
H3.3       <0.001             0.440              rho thông số

# Biểu đồ và lưu file

In [ ]:
def mean_ci(d, by):
    y_col = "sold_log" if "sold_log" in d.columns else Y
    g = d.groupby(by, observed=True)[y_col].agg(["mean", "std", "size"])
    g["ci"] = 1.96 * g["std"] / np.sqrt(g["size"])
    return g[g["size"] > 0]


def qgroup(x, k):
    """Chia nhóm theo GIÁ TRỊ (các giá trị trùng nhau luôn cùng một nhóm), đánh số 1..k'."""
    return "nhóm " + (pd.qcut(x, k, labels=False, duplicates="drop") + 1).astype("Int64").astype(str)


def draw(ax, g, title, xlabel):
    ax.errorbar(range(len(g)), g["mean"], yerr=g["ci"], fmt="o-", capsize=4, color="#1f6feb")
    ax.set_xticks(range(len(g)))
    ax.set_xticklabels([f"{i}\n(n={n})" for i, n in zip(g.index.astype(str), g["size"])], fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel("TB log(1+doanh số)", fontsize=8)
    ax.grid(alpha=.3)


#  Nhóm 1 
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ct = cat_tbl.sort_values("spearman_rho")
ax[0].barh(ct.ngành, ct.spearman_rho,
           color=np.where(ct.p_Holm < ALPHA, np.where(ct.spearman_rho < 0, "#1f6feb", "#d1495b"), "#bbbbbb"))
ax[0].axvline(0, color="k", lw=.8)
ax[0].set_title("H1.1 Spearman(giá tương đối, doanh số) theo ngành\n(xám = không có ý nghĩa sau Holm)", fontsize=10)
ax[0].tick_params(labelsize=8)

smp = [("Mẫu chính", wc), ("Bỏ Tiki Trading", wc_nt), ("Chỉ Tiki Trading", wc_tt)]
ax[1].errorbar(range(3), [s_["rho"] for _, s_ in smp],
               yerr=[[s_["rho"] - s_["lo"] for _, s_ in smp], [s_["hi"] - s_["rho"] for _, s_ in smp]],
               fmt="o", capsize=5, color="#1f6feb")
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xticks(range(3))
ax[1].set_xticklabels([f"{k}\n(n={s_['n']})" for k, s_ in smp], fontsize=8)
ax[1].set_title("H1.1 Spearman gộp trong ngành (CI 95%)", fontsize=10)
ax[1].grid(alpha=.3)
fig.tight_layout()
fig.savefig(OUT / "fig" / "nhom1.png", dpi=140)
plt.close(fig)

#  Nhóm 2 
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
g = mean_ci(rated.assign(b=qgroup(rated.rating_bayes, 5)), "b")
draw(ax[0], g, "H2.1 Doanh số theo nhóm rating_bayes (nhóm 1 thấp → cao)", "nhóm rating_bayes (mẫu >= 5 review)")

ax[1].errorbar(range(3), tier_tbl.spearman_rho,
               yerr=[tier_tbl.spearman_rho - [float(s_.split(' … ')[0]) for s_ in tier_tbl.ci95],
                     [float(s_.split(' … ')[1]) for s_ in tier_tbl.ci95] - tier_tbl.spearman_rho],
               fmt="o-", capsize=5, color="#d1495b")
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xticks(range(3))
ax[1].set_xticklabels([f"{t_}\n(n={n_})" for t_, n_ in zip(tier_tbl.nhóm_review, tier_tbl.n)], fontsize=8)
ax[1].set_title("H2.2 Spearman(rating, doanh số) theo số review", fontsize=10)
ax[1].grid(alpha=.3)

ax[2].errorbar(range(len(marg)), marg.độ_dốc_rating, yerr=[marg.độ_dốc_rating - marg.ci_thấp, marg.ci_cao - marg.độ_dốc_rating],
               fmt="o-", capsize=4, color="#d1495b")
ax[2].axhline(0, color="k", lw=.8)
ax[2].set_xticks(range(len(marg)))
ax[2].set_xticklabels(marg.số_review.astype(str), fontsize=8)
ax[2].set_xlabel("số review của sản phẩm", fontsize=8)
ax[2].set_title("H2.2 Tác dụng +1 sao rating theo số review\n(mô hình tương tác liên tục)", fontsize=10)
ax[2].grid(alpha=.3)
fig.tight_layout()
fig.savefig(OUT / "fig" / "nhom2.png", dpi=140)
plt.close(fig)

#  Nhóm 3 
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
draw(ax[0], mean_ci(core.assign(b=pd.cut(core.n_images, [0, 2, 4, 7, 12, np.inf], labels=["1-2", "3-4", "5-7", "8-12", "13+"])), "b"),
     "H3.1 Doanh số theo số ảnh", "số ảnh")
draw(ax[1], mean_ci(core.assign(b=core.has_video.map({0: "không video", 1: "có video"})), "b"), "H3.2 Doanh số theo video", "")
draw(ax[2], mean_ci(core.assign(b=pd.cut(core.n_specs, [0, 4, 7, 10, np.inf], labels=["1-4", "5-7", "8-10", "11+"])), "b"),
     "H3.3 Doanh số theo số thông số kỹ thuật", "số thông số")
fig.tight_layout()
fig.savefig(OUT / "fig" / "nhom3.png", dpi=140)
plt.close(fig)

#  Nhóm 4 
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
c4 = core.assign(b=pd.Categorical(core.seller_type, ["Seller thường", "Official khác", "Tiki Trading"], ordered=True))
draw(ax[0], mean_ci(c4, "b"), "H4.1 Doanh số theo loại người bán", "loại người bán")
draw(ax[1], mean_ci(core.assign(b=qgroup(core["s_avg_rating_point"], 4)), "b"),
     "H4.2 Doanh số theo điểm đánh giá của shop", "nhóm điểm shop")
draw(ax[2], mean_ci(core.assign(b=qgroup(core["s_total_follower"], 4)), "b"),
     "H4.3 Doanh số theo số người theo dõi", "nhóm người theo dõi")
fig.tight_layout()
fig.savefig(OUT / "fig" / "nhom4.png", dpi=140)
plt.close(fig)



_NUM = r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?"
def parse_stats(text):
    """Bóc tách chỉ số từ chuỗi chú thích."""
    out = []
    for part in str(text).split(";"):
        part = part.strip()
        m = re.match(rf"^([^=\d(][^=]*?)\s*=\s*({_NUM})", part)
        if m:
            out.append((m.group(1).strip(), float(m.group(2))))
            continue
        m = re.match(rf"^({_NUM})(?![\d.])", part)
        if m:
            out.append(("hệ_số", float(m.group(1))))
    return out


def parse_ci(text):
    """Trích xuất khoảng tin cậy 95%."""
    v = re.findall(_NUM, str(text))
    return (float(v[0]), float(v[1])) if len(v) == 2 else (np.nan, np.nan)


def export_numbers():
    cols = [
        "nhóm", "giả_thuyết", "loại", "vai_trò_hoặc_mô_hình", 
        "kiểm_định_hoặc_biến", "chỉ_số", "n", "giá_trị", "sai_số_chuẩn", 
        "ci_thấp", "ci_cao", "p_hai_phía", "p_Holm", "đạt", "kết_luận"
    ]
    rows = []

    def add(hid, loai, role, name, metric, n=np.nan, value=np.nan, se=np.nan, lo=np.nan, hi=np.nan, p=np.nan,
            p_holm=np.nan, ok=np.nan, concl=""):
        rows.append(dict(
            nhóm=int(hid[1]) if hid and hid[0] == "H" else np.nan, 
            giả_thuyết=hid, loại=loai, vai_trò_hoặc_mô_hình=role, kiểm_định_hoặc_biến=name, 
            chỉ_số=metric, n=n, giá_trị=value, sai_số_chuẩn=se, ci_thấp=lo, ci_cao=hi, 
            p_hai_phía=p, p_Holm=p_holm, đạt=ok, kết_luận=concl
        ))

    # Tổng hợp giả thuyết
    for _, r_ in S.iterrows():
        strong = {True: 1, False: 0}.get(r_.hiệu_ứng_không_yếu, np.nan)
        add(r_.id, "tổng_hợp", "kiểm định chính", r_.kiểm_định_chính, "p", r_.n, np.nan, p=r_.p_chính, p_holm=r_.p_chính_Holm,
            concl=r_.kết_luận_cuối)
        est = [float(x) for x in re.findall(_NUM, str(r_.ước_lượng_kiểm_chứng))]
        for k, v in enumerate(est or [np.nan]):
            add(r_.id, "tổng_hợp", "kiểm chứng", r_.kiểm_chứng, f"ước_lượng_{k + 1}" if len(est) > 1 else "ước_lượng", r_.n, v,
                p=r_.p_kiểm_chứng, p_holm=r_.p_kiểm_chứng_Holm)
        add(r_.id, "tổng_hợp", "hiệu ứng", r_.mức_ảnh_hưởng, "hiệu_ứng_đủ_mạnh(1=có,0=yếu)", r_.n, strong)

    #  Mọi phép kiểm định + Bảng phụ
    for sheet, blocks in REP.sheets.items():
        if not sheet.startswith("Nhóm"):
            continue
        for title, d_ in blocks:
            hid = title.split(" — ")[0].strip()
            if "vai_trò" in d_.columns and "kiểm_định" in d_.columns:
                for _, t_ in d_.iterrows():
                    lo, hi = parse_ci(t_.ci95)
                    vals = parse_stats(t_.thống_kê_hoặc_ước_lượng) or [("giá_trị", np.nan)]
                    if all(nm.startswith("p(") for nm, _ in vals):
                        for nm, v in vals:
                            add(hid, "kiểm_định", t_.vai_trò, t_.kiểm_định, nm, t_.n, p=v)
                        add(hid, "kiểm_định", t_.vai_trò, t_.kiểm_định, "p_kết_hợp(max)", t_.n, p=t_.p_hai_phía)
                        continue
                    for k, (nm, v) in enumerate(vals):
                        add(hid, "kiểm_định", t_.vai_trò, t_.kiểm_định, nm, t_.n, v, lo=lo if k == 0 else np.nan,
                            hi=hi if k == 0 else np.nan, p=t_.p_hai_phía if k == 0 else np.nan)
            else:
                label_cols = [c for c in d_.columns if not str(c).startswith("ci")
                              and not (pd.api.types.is_numeric_dtype(d_[c]) or pd.api.types.is_bool_dtype(d_[c]))]
                num_cols = [c for c in d_.columns if c not in label_cols and (pd.api.types.is_numeric_dtype(d_[c])
                                                                            or pd.api.types.is_bool_dtype(d_[c]))]
                for _, t_ in d_.iterrows():
                    label = " | ".join(str(t_[c]) for c in label_cols)
                    for c in num_cols:
                        add(hid, "bảng_phụ", title.split(" — ", 1)[-1], label, c, value=float(t_[c]) if pd.notna(t_[c]) else np.nan)

    # Độ vững qua các phiên bản
    for _, r_ in RB.iterrows():
        nums = [float(x) for x in re.findall(_NUM, str(r_.ước_lượng))] if "=" not in str(r_.ước_lượng) else []
        p_val = r_.get("p_hai_phía", r_.get("p_hai_phía_max", np.nan))
        for k, v in enumerate(nums or [np.nan]):
            add(r_.giả_thuyết, "độ_vững", r_.phiên_bản, "biến chính", f"hệ_số_{k + 1}" if len(nums) > 1 else "hệ_số",
                r_.n, v, p=p_val, ok=int(r_.đạt))

    # Mọi hệ số hồi quy
    for name, t_ in FULLREG:
        hid = name.split()[0]
        n_obs = float(t_.loc[t_["biến"] == "[n quan sát]", "hệ_số"].iloc[0])
        for _, c_ in t_[t_["biến"] != "[n quan sát]"].iterrows():
            add(hid, "hồi_quy", name, c_["biến"], "hệ_số", n_obs, c_["hệ_số"], se=c_["sai_số_chuẩn"], lo=c_["ci_2.5%"],
                hi=c_["ci_97.5%"], p=c_["p"])

    #  Mô hình tổng hợp chạy hai lần
    for _, j_ in JOINT.iterrows():
        add("MH", "mô_hình_tổng_hợp", f"{j_['mẫu']} | {j_['mô_hình']}", j_["biến"], "hệ_số_chuẩn_hóa", j_["n"], j_["hệ_số_chuẩn_hóa"],
            se=j_["sai_số_chuẩn"], lo=j_["ci_thấp"], hi=j_["ci_cao"], p=j_["p"])
    
    out = pd.DataFrame(rows, columns=cols)
    path = OUT / "ket_qua_so.csv"
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path, out


# Thực thi xuất file và báo cáo
_path, _num = export_numbers()
print(f"[Xuất số] {len(_num):,} dòng -> {_path.name} (loại: {_num['loại'].value_counts().to_dict()})")

REP.save(OUT / "tiki_hypothesis_results.xlsx")
print(f"\nĐã lưu thành công các file báo cáo vào thư mục: {OUT.resolve()}")

[Xuất số] 375 dòng -> ket_qua_so.csv (loại: {'hồi_quy': 171, 'bảng_phụ': 84, 'kiểm_định': 35, 'mô_hình_tổng_hợp': 34, 'tổng_hợp': 29, 'độ_vững': 22})

Đã lưu thành công các file báo cáo vào thư mục: E:\DA cơ bản\CK\Kiểm định giả thuyết\results
